# TP5 — CNNs para predicción de unión de JunD al ADN

                Notebook-esqueleto didáctico armado a partir de `TP5-CNNs.pdf`.

                **Cómo usar este notebook en equipo**
                - El texto de la consigna está integrado en el mismo orden del PDF, sin separarlo por páginas.
                - Debajo de cada bloque de consigna hay celdas **TODO** para completar de a pasos.
                - Las celdas TODO no resuelven el TP: tienen plantillas, nombres sugeridos y `raise NotImplementedError`.
                - Si se dividen tareas, una persona puede tomar datos/OHE, otra modelo, otra métricas, otra interpretación biológica.
                - Dejen las conclusiones escritas en las celdas de análisis: eso facilita integrar el trabajo grupal.

## 0. Setup general

                Ejecuten esta celda al inicio. Si alguna librería falta, actualicen el entorno Conda antes de resolver el TP.

                Nota: para métricas como ROC, PR, AUC o F1 pueden usar `scikit-learn`; si no está instalado, agréguenlo al entorno o implementen las métricas manualmente.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Imports opcionales para métricas. Descomenten si instalan/usan scikit-learn.
# from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
# from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

# Rutas relativas pensadas para ejecutar desde notebooks/TP5-CNNs.
# Si su Jupyter usa otro working directory, ajusten PROJECT_ROOT.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "TP5-CNNs":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

BED_PATH = PROJECT_ROOT / "jun_np_chr22_GRCh38.bed"
POSITIVE_FASTA_PATH = PROJECT_ROOT / "sequences_upper.fasta"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("BED:", BED_PATH)
print("FASTA positivos:", POSITIVE_FASTA_PATH)


## Consigna — TP5 y 5.1

Trabajo Práctico 5
Convolutional Neural Networks (CNNs)
Objetivo: El objetivo del presente TP es familiarizarse con las CNNs, particularmente sus dos
componentes  principales  los  “convultional  filters”  y  las  capas  de  “pooling”.  Para  ello
avanzaremos  en  el  análisis  de  secuencias  buscando  predecir  la  unión  de  un  factor  de
transcripción (JunD) al ADN.
JunD  es un miembro de  la familia  de  factores de transcripción  AP-1 (activator protein-1),
conocida por su papel crucial en la regulación de la expresión génica asociada a procesos
como la proliferación celular, diferenciación y respuesta al estrés. Al igual que otros miembros
de la familia Jun (c-Jun y JunB), JunD forma heterodímeros con proteínas de la familia Fos o
ATF para unirse a regiones específicas del ADN conocidas como sitios TRE (TPA-response
elements)  o  CRE  (cAMP  response  elements).  Sin  embargo,  JunD  se  distingue  por  sus
funciones más específicas y su menor capacidad para promover la proliferación celular, lo que
sugiere un papel más prominente en la regulación de procesos homeostáticos y protectores
contra el estrés oxidativo. Además, se ha vinculado a la regulación de genes implicados en la
supervivencia celular y en la prevención de daño genómico, destacando su importancia en
contextos fisiológicos y patológicos, como el cáncer y las enfermedades neurodegenerativas.
5.1 secuencias donde se une el FT (True Positive set).
Los sitios de unión a un factor de transcripción (TF) se estudian experimentalmente mediante
experimentos de ChIPSeq. El consorcio ENCODE ha realizado en las últimas décadas estudios
sistemáticos de este tipo, así como también de accesibilidad de la cromatina, para una multitud
de FT en diferentes tipos y lineas celulares. El resultado final del un análisis bioinformático de
un experimento de ChIPSeq se resume en un archivo BED que contiene las coordenadas
(Chrom:pos) de cada “pico” (sitio de unión) obtenido.
En este caso el archivo que bajamos de encode: jun_np_chr22_GRCh38.bed , posee los 
picos para el cromosoma 22. A continuación se muestran las 2 primeras lineas del mismo 
chrom  pos_i   pos_f   .       1000    .       val     -1 nada    size
chr22   20777767        20778110        .       1000    .       363.72008   -1 3.79036 178
Las primeras tres columnas son obvias, luego sigue un “.”, que al igual que el otro “.” el “-” y la 
columna que bautice “nada” no se utilizan en un BED de este tipo. El valor de 1000 es la 
columna que utilizan los “Geonome Browsers” para visualizar el pico (y tampoco nos importa), 
finalmente el valor que bautizamos “val” es la INTENSIDAD el pico (o sea la señal).
Nota: En realidad los BED de encode NO tienen una primer fila, pero aquí fue agregada a 
manos por los docentes para simplificar el TP.

Por otro lado tenemos la secuencia del Chr22 del Genoma de ref versión GRCh38 en formato 
fasta (chr22_GRCh28.fasta) para poder obtener las secuencias que reconoce el TF.
Ejercicio 5.1 (opcional). Escriba un código que permita a partir del BED y el fasta obtener las
secuencias donde el FT se une al ADN (también puede obtenerlas con samtools). Analicé
además la distribución de intensidad de los picos del BED y del tamaño de la secuencia
reportada. Vaya a alguno de los genome browsers cargue el BED y vea si puede visualizar en
el mismo los picos.
En caso de que haya realizado el ejercicio 0 usted ya posee sus secuencias positivas, en caso 
contrario, el archivo sequences_upper.fasta contiene las mismas ya extraídas por el docente.

### TODO 5.1 opcional — Explorar BED y FASTA

                Completen esta parte si quieren reconstruir/validar las secuencias positivas desde el BED y el genoma.

                Tareas sugeridas:
                - Leer `jun_np_chr22_GRCh38.bed` en un `DataFrame`.
                - Revisar intensidades de pico (`val`) y tamaños (`size`).
                - Graficar distribuciones.
                - Opcional: extraer secuencias desde un FASTA genómico si lo tienen disponible.
                - Opcional: cargar el BED en un genome browser y registrar observaciones.

In [ ]:
# TODO 5.1.a: leer el BED y revisar columnas.
bed_df = None

# TODO 5.1.b: graficar distribución de intensidad de picos.
# plt.hist(...)

# TODO 5.1.c: graficar distribución de tamaños de picos/secuencias.
# plt.hist(...)

# TODO 5.1.d opcional: escribir funciones para parsear FASTA genómico y extraer secuencias positivas.
# def parse_fasta_genomico(path):
#     ...
#
# def extraer_secuencias_positivas(bed_df, fasta_genomico):
#     ...

raise NotImplementedError("Completar exploración opcional del BED/FASTA.")


### Análisis 5.1

                Escriban acá:
                - ¿Cómo se distribuye la intensidad de los picos?
                - ¿Cómo se distribuyen los tamaños?
                - Si miraron un genome browser, ¿qué observaron?

## Consigna — 5.2

5.2 Preparando el input para la CNN
Para terminar de preparar nuestro input para la CNN necesitamos: 1) los negativos, 2) pasar 
las secuencias a one-hot-encoding (OHE), y 3) definir el padding.
Para  los  negativos  podemos  utilizar  los  códigos  del  TP  anterior  y  generar  N1  un  set  de
negativos de secuencias al azar, N2 un set de negativos haciendo permutaciones de 2nt (o 3)
de los Positivos, y en este caso aregaremos uno (o varios) N3 que comprenden secuencias del
mismo cromosoma donde el FT asumimos no se une.  
Si recordamos el TP anterior, cuando pasamos una secuencia de ADN a OHE, lo que hacemos
es convertir cada base (A,C,G,T) a un vector de 4 posiciones donde la posición del 1 es la que
corresponde a la base, y luego concatenamos estos vectores uno detrás de otro. De este modo
si tenemos una secuencia de 10nt obtenemos un tensor de dimensión 40. Cuando usamos
CNNs, como el input puede tener “N” canales (x ejemplo en imágenes son las intensidades de
los 3 colores RGB) podemos asignar cada base a un “canal” y entonces nuestra secuencia de
10nt se convierte en un tensor de dimensión [4,10].
Finalmente, el padding. Este consiste en agregar inputs “nulos” para completar el campo visual
o dimensión del input requerido por la red. Dicho de otro modo, si nuestra red espera un input
de 4x100 (4 canales x 100 nucleótidos) y tenemos una secuencia de longitud 98 dim=[4,98)
¿que hacemos?. El padding consiste en agregar en uno (o ambos) extremos inputs con 0s para
completar la dimensión del input requerido. O sea en este caso deberíamos agregar 2 inputs
mas en la dimensión 2 (como si fueran 2 nt mas) con todos 0, para que la dimensión final sea
[4,100]. En este caso como hay que agregar 2 se puede agregar uno de cada lado.
Ejercicio 5.2: a) genere los 3 sets de secuencias negativas. Para el conjunto N3 (secuencias
del  genoma que  NO unen  el FT) puede  en  primer lugar tomar secuencias  (de largo  fijo)
adyacentes  y/o  a  “n”  nt  de  distancia  del  sitio  de  unión  del  FP.  Alternativamente  puede
seleccionar secuencias del Chr22 al azar (o puede probar ambas y ver como esta variable
afecta el modelo).
b) escriba el código para a partir del fasta con las secuencias de ADN, los convierta en un
tensor de pytorch de dimension Nseqs, 4, largo Seq

c) integre al codigo “b” la opción de hacer “0” padding y/o recorte las secuencias a un largo fijo.
d)  Escriba  el  código  que  genera  ademas  el  tensor  que  posee  la  clasificación  real  de  la
secuencia en si une (y_true=1) (o no, y_true=0) el FT. Usualmente a este tensor se lo llama
y_true.
Nota:  recomendamos  manejar  los  datos  en  un  DF  que  permita  además  de  realizar  las
conversiones mencionadas separar el dataset en entrenamiento y validación.
Debería obtener algo de este estilo:

### TODO 5.2.a — Generar sets negativos N1, N2 y N3

                Completen funciones para generar:
                - **N1:** secuencias negativas al azar.
                - **N2:** negativos por permutaciones de 2 nt o 3 nt de positivos.
                - **N3:** secuencias del cromosoma 22 donde asumimos que JunD no se une.

                Importante: documenten qué estrategia usan para N3, porque puede afectar la dificultad del problema y la performance del modelo.

In [ ]:
# TODO: leer secuencias positivas desde sequences_upper.fasta.
positive_sequences = None


def leer_fasta(path):
    """Devuelve una estructura con ids y secuencias desde un FASTA."""
    raise NotImplementedError


def generar_negativos_azar(n_seqs, largo, rng=None):
    """Genera N1: secuencias de ADN al azar."""
    raise NotImplementedError


def generar_negativos_shuffle(sequences, k=2, rng=None):
    """Genera N2: negativos por permutación en bloques de k nucleótidos."""
    raise NotImplementedError


def generar_negativos_genomicos(bed_df, estrategia="adyacente", largo=100, rng=None):
    """Genera N3: negativos del chr22 por estrategia adyacente/distante/azar."""
    raise NotImplementedError


# TODO: construir n1_sequences, n2_sequences y n3_sequences.
# n1_sequences = ...
# n2_sequences = ...
# n3_sequences = ...


### TODO 5.2.b y 5.2.c — One-hot encoding, padding y recorte

                El input esperado para una CNN 1D en PyTorch será un tensor con forma:

                `(N_seqs, 4, largo_seq)`

                Completen:
                - Conversión A/C/G/T a canales.
                - Padding con ceros si la secuencia es corta.
                - Recorte si la secuencia es larga.
                - Validaciones de dimensión.

In [ ]:
BASE_TO_CHANNEL = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3,
}


def ajustar_largo(seq, nt_len, modo="centrado"):
    """Aplica padding o recorte para obtener secuencias de largo fijo nt_len."""
    raise NotImplementedError


def one_hot_encode_sequence(seq, nt_len):
    """Convierte una secuencia a un array OHE de forma (4, nt_len)."""
    raise NotImplementedError


def secuencias_a_tensor(sequences, nt_len):
    """Convierte una lista de secuencias a tensor PyTorch (N, 4, nt_len)."""
    raise NotImplementedError


# TODO: elegir largo fijo de entrada para los primeros modelos.
nt_len = None

# TODO: transformar positivos y negativos a tensores.
# X_pos = ...
# X_neg = ...


### TODO 5.2.d — Dataset, etiquetas y split entrenamiento/validación

                Armen un `DataFrame` maestro que permita rastrear:
                - `sequence`
                - `label` (`1` positivo, `0` negativo)
                - `set_type` (`TP`, `N1`, `N2`, `N3`)
                - metadatos disponibles del BED, como intensidad del pico

                Después separen entrenamiento y validación/test.

In [ ]:
# TODO: construir dataset_df integrando positivos y negativos.
dataset_df = None

# TODO: crear y_true como tensor de etiquetas.
# y_true = ...

# TODO: separar train/test o train/validation.
# train_df = ...
# test_df = ...

# TODO: convertir splits a tensores.
# train = ...
# train_y = ...
# test = ...
# test_y = ...

raise NotImplementedError("Completar dataset, etiquetas y split.")


### Análisis 5.2

                Escriban acá:
                - ¿Cuántas secuencias positivas y negativas quedaron?
                - ¿Qué largo fijo eligieron y por qué?
                - ¿Qué estrategia usaron para N3?
                - ¿El dataset quedó balanceado?

## Consigna — 5.3

5.3 Armando la CNN
Ahora vamos a proceder a definir y entrenar la CNN
La primer CNN basica (provista gentilmente por los docentes) es la siguiente:
# Define the CNN model
import torch
import torch.nn as nn
import torch.nn.functional as F
class DNA_CNN(nn.Module):
   def __init__(self):
       super(DNA_CNN, self).__init__()
       
  c=8
       k=5
      self.conv1 = nn.Conv1d(in_channels=4, out_channels=c, kernel_size=k)
       # Fully connected layer: Maps to the output
       # Calculate the output size after the convolutional layer
       conv_output_size = c * (nt_len - k + 1)  # (L - K + 1) formula

self.fc1 = nn.Linear(conv_output_size, 1)  # Maps to 1 output
(binary classification)
  
   def forward(self, x):
       # Apply convolution, followed by ReLU activation
       x = F.relu(self.conv1(x))
      
       # Flatten the output from the convolutional layer
       x = x.view(x.size(0),  -1)   #  Reshape  to  (batch_size,
flattened_features)
      
       # Fully connected layer and output
       x = self.fc1(x)
       x = torch.sigmoid(x)  # Apply sigmoid for binary classification
       return x
Este código define una red neuronal convolucional (CNN) para realizar tareas de clasificación
binaria en datos secuenciales, como secuencias de ADN. A continuación se explica cada parte:
En primer lugar importamos las bibliotecas de pytorch (general, o sea torch),  torch.nn, que
proporciona  módulos  y  capas  para  construir  redes  neuronales  y   torch.nn.functional  que
contiene funciones como activaciones y operaciones que no son módulos.
La CNN esta empaquetada en una clase (DNA_CNN), que “hereda” la funcionalidad de la clase
de pytorch para construir redes, y permite como ya vimos en el TP anterior el manejo de los
modos forw, back y la actualización de parámetros.
Luego el  def __init__(self) es  un método especial en Python que se ejecuta automáticamente
cuando se crea una instancia de la clase 
super()  es una función en Python que permite acceder a métodos y atributos de la clase base
(en este caso, nn.Module).
Estos comandos aseguran que el modelo tenga acceso a las funcionalidades de nn.Module y la
red funcione (por ahora esto usted lo dejara fijo en bloque).
Ahora lo interesante...
La primer capa de la red que llamamos self.conv1 se define a partir del modulo correspondiente
en pytroch con los siguientes parámetros: in_channels, out_channels y kernel_size
in_channels=, son 4: Número de canales de entrada (por ejemplo, 4 bases de ADN: A, C, G, T).
out_channels: Número de filtros (o kernels) que generará la capa y 
kernel_size=k es el campo visual ( Tamaño de la ventana de filtro en nucleotidos).

La(s)  salidas  de  la  capa  convolucional  se  mapean  a  una  capa  “fully  connected” como  la
utilizada en el TP anterior que llamamos self.c1, y que entrega 1 solo valor como salida. (¿por
qué sólo 1 valor de salida?)
El segundo bloque es el que define el método forward (que es REQUERIDO por la clase
torch.nn.Module y que específica como fluyen los datos por la red.
En este caso primero se  aplica la convolución y la función de activación ReLU para introducir
no linealidad, luego la salida es “aplanada” (o sea se transforma a un tensor de dim[batch_size,
cnn_out] con  x.view(x.size(0), -1
En tercer lugar las características extraídas por la convolución son condensadas en una capa
lineal  para luego pasar por la función de activación  sigmoide  para obtener probabilidades
(salida entre 0 y 1).
Ejercicio 5.3 Pruebe que el código y su método de OHE “funcionan” inicializando la red y 
pidiéndole que procese un btach de input:
my_nn = DNA_CNN()
output = my_nn(test)
Analice brevemente la salida (en cuanto a valores, dimension). Si quiere, para tener como 
referencia, aproveche y calcule los parametros de calidad del modelo, como ser la “Confusión 
Matrix”, la Accuracy (o el F1-Score) y las curvas ROC / PR y su AUC. ¿que piensa sobre los 
resultados de este modelo “sin” entrenar?.

### TODO 5.3 — Definir la CNN base

                Usen el código provisto por la consigna como punto de partida.

                Completen una clase `DNA_CNN` que permita definir:
                - número de filtros convolucionales,
                - tamaño del kernel,
                - largo de entrada,
                - capa lineal final para clasificación binaria.

In [ ]:
class DNA_CNN(nn.Module):
    def __init__(self, nt_len, n_filters=8, kernel_size=5):
        super(DNA_CNN, self).__init__()

        # TODO: guardar hiperparámetros si les resulta útil.
        # self.nt_len = ...
        # self.n_filters = ...
        # self.kernel_size = ...

        # TODO: definir capa convolucional.
        # self.conv1 = nn.Conv1d(in_channels=4, out_channels=..., kernel_size=...)

        # TODO: calcular tamaño de salida luego de la convolución.
        # conv_output_size = ...

        # TODO: definir capa fully connected final.
        # self.fc1 = nn.Linear(conv_output_size, 1)

        raise NotImplementedError("Completar __init__ de DNA_CNN.")

    def forward(self, x):
        # TODO: aplicar convolución + ReLU.
        # x = ...

        # TODO: aplanar salida.
        # x = ...

        # TODO: capa fully connected + sigmoid.
        # x = ...
        return x


### TODO 5.3 — Probar modelo no entrenado

                Inicialicen la red y pídanle que procese un batch de input.

                Analicen:
                - dimensiones de salida,
                - rango de valores,
                - métricas de calidad sin entrenamiento.

In [ ]:
# TODO: inicializar modelo base.
my_nn = None

# TODO: elegir un batch de prueba.
# batch_test = ...

# TODO: obtener salida del modelo.
# output = my_nn(batch_test)

# TODO: inspeccionar valores y dimensiones.
# print(output.shape)
# print(output[:10])

# TODO opcional: calcular matriz de confusión, Accuracy, F1, ROC/PR y AUC.

raise NotImplementedError("Completar prueba del modelo no entrenado.")


### Análisis 5.3

                Escriban acá:
                - ¿Qué forma tiene la salida?
                - ¿Qué valores produce una red sin entrenar?
                - ¿Qué significan las métricas de un modelo no entrenado?

## Consigna — 5.4

5.4 El ciclo de entrenamiento
Ahora debemos armar el ciclo de entrenamiento del modelo, cuyo “posible” código se muestra 
a continuación
my_nn = DNA_CNN()
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(my_nn.parameters(), lr=0.001)
dataset = TensorDataset(train, y_true)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)
# Train the network
epochs = 100  # Number of epochs
for epoch in range(epochs):
   total_loss = 0
   for batch_X, batch_y in dataloader:
       # Forward pass
       output = my_nn(batch_X)  # Predict the output

#loss = criterion(output, batch_y)  # Compute the loss
       loss = criterion(output.squeeze(), batch_y.float())
       #loss = criterion(outputs.squeeze(), labels.float())
       # Backward pass
       optimizer.zero_grad()  # Zero the gradients
       loss.backward()        # Compute gradients
       optimizer.step()       # Update weights
       total_loss += loss.item()
   # Print loss every 10 epochs
   if epoch== 0 or (epoch + 1) % 10 == 0:
       test_out = my_nn(test) 
       test_loss = criterion(test_out.squeeze(), test_y.float())
       print(f"Epoch [{epoch+1}/{epochs}], Loss: 
{total_loss/len(dataloader):.4f}, test Loss: {test_loss:.4f}")
El mismo tiene dos secciones:
En la primera, luego de iniciar el modelo, se definen las caracteristicas del entrenamiento, a 
saber: i) la   función de pérdida como la Binary Cross Entropy Loss (BCELoss), ii) el optmizador
(en este caso Adam con un larning rate (lr) pequeño, y los datos. En este caso utilizamos 
primero una función que convierte los sets en tensores de pytorch, y luego usamos el dataloaer
para definir los batch. 
En la segunda se define el ciclo de entrenamiento (# de épocas =100), y en cada ciclo:
i) forward pass (obtención del output), 
ii) calculo de de la pérdida comparando las predicciones del modelo con las etiquetas 
verdaderas 
iii) Backward pass (primero se hacen zero los gradientes)
iv)   Actualiza los parámetros del modelo utilizando los gradientes calculados.
Note que todos pasos se realizan con comandos/funciones de pytorch pre-armadas por lo que 
solo hay que llamarlos y pytorch hace la magia.

Finalmente, se calcula el total de la LF para la época y se muestra la misma. En este caso 
como para monitorear el modelo de manera más precisa le pedimos que nos muestre la LF no 
solo en el conjunto de entrenamiento, sino también en el de testing/validación.
Si todo anduvo bien una prueba del ciclo debería mostrar algo asi:
Epoch [1/100], Loss: 0.6863, test Loss: 0.6817
Epoch [10/100], Loss: 0.4263, test Loss: 0.5994
Epoch [20/100], Loss: 0.2530, test Loss: 0.5100
etc.
Ejercicio 5.4 Utilizando los parámetros sugeridos (#filtros 8, dimesión 5 y con secuencias de
100nt),  realice  el  entrenamiento  de  la  DNA-CNN.  Obtenga  y  compare  para el  modelo  no
entrenado, y para su entrenamiento en 10, 50 y 100 epocas, la LF, la matriz de confusión, la
Accuracy, el F1-score y las curvas ROC y PR junto a su AUC. Compare si todas las métricas
siguen la misma tendencia (o no).

### TODO 5.4 — Ciclo de entrenamiento

                Completen una función de entrenamiento basada en el código sugerido por la consigna:
                - `BCELoss`,
                - optimizador Adam,
                - `TensorDataset`,
                - `DataLoader`,
                - loop por épocas,
                - guardado de loss de entrenamiento y validación.

In [ ]:
def entrenar_modelo(model, train, train_y, test, test_y, epochs=100, batch_size=10, lr=0.001):
    """Entrena una CNN y devuelve modelo entrenado e historial de pérdidas."""
    # TODO: definir criterion.
    # criterion = ...

    # TODO: definir optimizer.
    # optimizer = ...

    # TODO: armar TensorDataset y DataLoader.
    # dataset = ...
    # dataloader = ...

    # TODO: loop de entrenamiento.
    # history = {"epoch": [], "train_loss": [], "test_loss": []}
    # for epoch in range(epochs):
    #     ...

    raise NotImplementedError("Completar función de entrenamiento.")


# TODO: entrenar modelos de 10, 50 y 100 épocas con parámetros sugeridos.
# modelos = {}
# historiales = {}


### TODO 5.4 — Métricas comparativas

                Comparen:
                - modelo no entrenado,
                - entrenamiento en 10 épocas,
                - entrenamiento en 50 épocas,
                - entrenamiento en 100 épocas.

                Métricas pedidas:
                - LF/loss,
                - matriz de confusión,
                - Accuracy,
                - F1-score,
                - curvas ROC y PR con AUC.

In [ ]:
def evaluar_modelo(model, X, y, threshold=0.5):
    """Calcula predicciones, scores y métricas para un modelo."""
    # TODO: pasar modelo a eval, desactivar gradientes y predecir.
    # TODO: convertir scores a clases usando threshold.
    # TODO: calcular métricas.
    raise NotImplementedError


def graficar_curvas_roc_pr(resultados):
    """Grafica curvas ROC y PR para uno o más modelos."""
    raise NotImplementedError


# TODO: evaluar modelos no entrenado / 10 / 50 / 100 épocas.
# resultados_54 = ...

raise NotImplementedError("Completar evaluación comparativa del Ejercicio 5.4.")


### Análisis 5.4

                Escriban acá:
                - ¿Todas las métricas siguen la misma tendencia?
                - ¿Hay señales de sobreentrenamiento?
                - ¿Qué métrica les parece más informativa para este problema?

## Consigna — 5.5

5.5 Primeras CNNs
Como  ya  sabemos  la  clave  en  una  CNN  son  sus  filtros  convolucionales.  En  nuestra  red
podemos  definir  2  “hyper”  parámetros  que  son  el  numero  de  filtros  convolucionales  y  su
dimensión. Además tenemos cómo parámetro adicional el largo de la secuencia de entrada.
Nuestro objetivo en esta etapa es analizar el impacto de estos parámetros en la performance
de la red.
En segundo lugar vamos a realizar una interpretación de los filtros ya entrenados. Recordemos
que cada filtro de dimensión k, es una matriz de pesos (w) de dimensión  [canales, k]  que
multiplica las entradas. Entonces podemos observar los w para cada canal (A,C,T,G) para cada
una de las posiciones del filtro (1 a k) y eso nos mostrará que aprendió el mismo. 
El comando “  my_nn.conv1.weight.data “ permite obtener el tensor de los filtros con dimensión
[#n  filters,  in_channels,  kernel_size])  y  luego  utilizando  los  indices  podemos  analizar  el
resultado en formato tabla o gráficamente.
Ejercicio 5.5. a) Explore como se modifica la performance de la red al variar el número de filtros
(valores sugeridos 4,8, 12 y 16) y su dimensión (K=3, 5, 10, 15). Analice los resultados.
b) Elija un tamaño de filtro, y visualice los filtros entrenados, seleccione un filtro y analicé el
resultado.

A continuación le dejamos una imagen de ejemplo:

### TODO 5.5.a — Explorar número de filtros y tamaño de kernel

                Prueben combinaciones sugeridas:
                - número de filtros: `4, 8, 12, 16`
                - tamaños de kernel: `3, 5, 10, 15`

                Guarden performance en una tabla para comparar.

In [ ]:
filter_grid = [4, 8, 12, 16]
kernel_grid = [3, 5, 10, 15]

# TODO: entrenar/evaluar una CNN por combinación.
# resultados_grid = []
# for n_filters in filter_grid:
#     for kernel_size in kernel_grid:
#         ...

# TODO: convertir resultados a DataFrame.
# grid_df = pd.DataFrame(resultados_grid)

# TODO: graficar heatmaps o curvas comparativas.

raise NotImplementedError("Completar exploración de filtros y kernel_size.")


### TODO 5.5.b — Visualizar filtros entrenados

                `my_nn.conv1.weight.data` permite obtener un tensor de dimensión:

                `[# filtros, in_channels, kernel_size]`

                Completen funciones para visualizar pesos de filtros como tabla o gráfico.

In [ ]:
def obtener_filtros(model):
    """Devuelve pesos de conv1 como array/tensor para análisis."""
    raise NotImplementedError


def graficar_filtro(model, filter_idx, channel_labels=("A", "C", "G", "T")):
    """Grafica un filtro entrenado como matriz canales x posiciones."""
    raise NotImplementedError


# TODO: elegir modelo entrenado y filtro a visualizar.
# filtros = ...
# graficar_filtro(...)

raise NotImplementedError("Completar visualización e interpretación de filtros.")


### Análisis 5.5

                Escriban acá:
                - ¿Qué combinación de filtros/kernel funcionó mejor?
                - ¿Qué aprendió el filtro que eligieron visualizar?
                - ¿Se ve algún patrón compatible con un motivo de ADN?

## Consigna — 5.6

5.6 El Max pooling
Agreguemos ahora una capa de Maxpooling. 
El max pooling tiene 2 efectos. Por un lado, permite “reducir” la dimensionalidad, lo que hace la
red más rápida y que se requieran menos parámetros en las capas futuras, y genera modelos
más robustos al tolerar pequeñas variaciones en los datos de entrada.  
A continuación se muestra el codigo modificado
 
class DNA_CNN_maxpool(nn.Module):
   def __init__(self):
       super(DNA_CNN_maxpool, self).__init__()
       c = 8  # Number of output channels for the convolutional layer
       k = 5  # Kernel size for the convolutional layer
       pool_size = 2  # Kernel size for max pooling
      
      self.conv1 = nn.Conv1d(in_channels=4, out_channels=c, kernel_size=k)
       self.pool = nn.MaxPool1d(kernel_size=pool_size)  # Max pooling
layer
      
       # Calculate the output size after convolution and pooling
       conv_output_length = (100 - k + 1)  # Length after convolution (L -
K + 1)

pooled_output_length = conv_output_length // pool_size  # Length
after pooling
      
       conv_output_size = c * pooled_output_length  # Total flattened size
       self.fc1 = nn.Linear(conv_output_size, 1)  # Fully connected layer
  
   def forward(self, x):
       # Apply convolution, ReLU, and max pooling
       x = F.relu(self.conv1(x))  # Convolution + ReLU activation
       x = self.pool(x)  # Apply max pooling
      
       # Flatten the output from the pooling layer
   x = x.view(x.size(0),  -1)   #  Reshape  to  (batch_size,
flattened_features)
      
       # Fully connected layer and output
       x = self.fc1(x)
       x = torch.sigmoid(x)  # Apply sigmoid for binary classification
       return x
El primer cambio es el agregado de la “pooling layer” luego de la convolucional
self.pool = nn.MaxPool1d(kernel_size=pool_size)
En este caso el kernel_size/pool_size específica la dimensión (ventana) sobre la que se hará el
pooling (en este caso 2)
En el presente caso, las ventanas de pooling son adyacentes (NO superpuestas) lo que resulta 
en una reducción de la dimensionalidad. En este caso como pool_size es 2, cada 2 posiciones 
se hace el max pool por lo que la dimensión es reducida a la mitad. Por lo tanto se debe 
recalcular el tamaño de la salida, que es el input de la fully connected layer.
Finalmente, en la definición del forward(self,x) debemos también agregar la capa de 
max.polling
Alternativamente, si no se quiere reducir la dimensión, se puede utilizar un stride de 1 en la 
pooling layer => self.pool = nn.MaxPool1d(kernel_size=pool_size, stride=1) 
Nota: en todos los casos cuando utilice el pooling debe ser cuidadoso en como de acuerdo al 
pool_size, stride y padding, resulta la dimensión del tensor de salida, ya que este define el 
tamaño de la ultima capa.

Ejercicio 5.6. Agregue a su DNA_CNN una pollin layer y analice el efecto de la misma en la
performance del modelo. Pruebe variar el poolin_size y el stride. 
A diferencia de los modelos para reconocer imágenes donde el pooling puede ser sumamente
efectivo, en ADN no se ha visto que el mismo lo sea ¿que le muestran sus datos?

### TODO 5.6 — CNN con Max Pooling

                Agreguen una capa `MaxPool1d` después de la convolución.

                Tengan cuidado con:
                - `pool_size`,
                - `stride`,
                - cálculo del tamaño final antes de `Linear`.

In [ ]:
class DNA_CNN_MaxPool(nn.Module):
    def __init__(self, nt_len, n_filters=8, kernel_size=5, pool_size=2, stride=None):
        super(DNA_CNN_MaxPool, self).__init__()

        # TODO: definir conv1.
        # self.conv1 = ...

        # TODO: definir pool.
        # self.pool = nn.MaxPool1d(kernel_size=pool_size, stride=stride)

        # TODO: calcular largo luego de conv y pooling.
        # conv_output_length = ...
        # pooled_output_length = ...
        # conv_output_size = ...

        # TODO: definir fc1.
        # self.fc1 = ...

        raise NotImplementedError("Completar CNN con MaxPool.")

    def forward(self, x):
        # TODO: conv + ReLU + pooling + flatten + fc + sigmoid.
        return x


# TODO: entrenar modelos variando pool_size y stride.
# pool_results = ...

raise NotImplementedError("Completar análisis de Max Pooling.")


### Análisis 5.6

                Escriban acá:
                - ¿El pooling mejora o empeora la performance?
                - ¿Qué pasa al variar `pool_size`?
                - ¿Qué pasa al variar `stride`?
                - ¿Los datos apoyan la idea de que el pooling no siempre ayuda en ADN?

## Consigna — 5.7

5.7 Análisis e interpretación de las predicciones de CNN_DNA
Para  realizar  un  análisis  final  “más  biológico”  de  nuestra  CNN_DNA  vamos  a  realizar  2
actividades. En primer lugar realizaremos un análisis del modelo en relación con los datos
experimentales,  particularmente  la  intensidad  del  pico,  y  en  segundo  lugar  analizaremos
utilizando “masking” que regiones de las secuencias son las que el FT reconoce con más
fuerza, y veremos si con las mismas podemos generar un MSA y un logo.
El masking consiste en elegir una (o más) posiciones de la secuencia, enmascararlas (o sea
hacer 0 todos sus inputs) y evaluar la secuencia enmascarada con la red. Si tomamos una
secuencia  de  referencia  que  tenga  un  “score”  alto,  y  enmascaramos  de  a  uno  (o  más)
nucleótidos, podemos analizar cuanto cambia (cae) el valor predicho. 
A continuación, a modo de ejemplo, se muestra un gráfico de como cambia el score de la red al
ir enmascarando las posiciones sucesivas (en ventanas de 5nt)
Ejercicio 5.7. Seleccione alguna (la mejor?) de las redes entrenadas en los ejercicios previos.
a) obtenga la prediccion para todas las secuencias TP del Chr22, compare el score de las
predicciones de cada secuencia con la intensidad del pico correspondiente en el BED. ¿Que le
parece los resultados?
b) Escriba un codigo que permita enmascarar regiones de la secuencia y evaluar (como en el
ejemplo) el impacto del mismo sobre la prediccion de la red. Utilice esta herramienta para

seleccionar las regiones de 10-25nt de longitud de cada secuencia que son màs importantes
para el modelo. 
c) Obtenga las regiones mas relevantes y realice un MSA (puede utilizar un programa externo
online o aprovechar sus conocimientos de BioinfoAv para hacerlo en python). Luego obtenga y
analice el logo del MSA correspondiente.
d) (opcional) Realice el análisis de secuencias relevantes-MSA-logo utilizando una red con
pocos filtros (4?) y un kernel de imension similar al de las secuencias a extraer (i.e 5-15nt),
compare los parámetros del filtro con el logo. ¿Coinciden?

### TODO 5.7.a — Predicciones TP vs intensidad del pico

                Usen la mejor red entrenada para predecir scores sobre todas las secuencias positivas del chr22.

                Comparen el score del modelo con la intensidad experimental del pico en el BED.

In [ ]:
# TODO: elegir mejor modelo entrenado.
best_model = None

# TODO: obtener scores para todas las secuencias TP.
# tp_scores = ...

# TODO: unir scores con intensidad del BED.
# tp_analysis_df = ...

# TODO: graficar score CNN vs intensidad de pico.
# plt.scatter(...)

raise NotImplementedError("Completar comparación de scores con intensidad de pico.")


### TODO 5.7.b — Masking de regiones

                El masking consiste en hacer cero una o más posiciones de la secuencia y evaluar cuánto cambia el score del modelo.

                Completen una herramienta para:
                - enmascarar ventanas,
                - medir caída de score,
                - elegir regiones importantes de 10–25 nt.

In [ ]:
def maskear_region(ohe_seq, start, window_size):
    """Devuelve una copia de la secuencia OHE con una ventana enmascarada en cero."""
    raise NotImplementedError


def escanear_masking(model, ohe_seq, window_size=5):
    """Evalúa caída de score al enmascarar ventanas sucesivas."""
    raise NotImplementedError


def seleccionar_region_importante(masking_result, min_len=10, max_len=25):
    """Selecciona región relevante usando el perfil de masking."""
    raise NotImplementedError


# TODO: aplicar masking a secuencias con score alto.
# masking_profiles = ...

raise NotImplementedError("Completar análisis de masking.")


### TODO 5.7.c y 5.7.d — Regiones relevantes, MSA, logo y filtros

                Completen el flujo final:
                - extraer regiones relevantes,
                - realizar MSA con herramienta externa o Python,
                - generar logo,
                - opcional: comparar logo con filtros de una red chica.

In [ ]:
# TODO: guardar regiones relevantes en FASTA para MSA externo, si corresponde.
def escribir_fasta_regiones(regiones, output_path):
    raise NotImplementedError


# TODO: cargar/representar el MSA resultante.
# msa = ...

# TODO: generar logo o matriz de frecuencias.
# logo_df = ...

# TODO opcional: comparar logo con pesos de filtros.

raise NotImplementedError("Completar MSA/logo e interpretación final.")


### Análisis 5.7

                Escriban acá:
                - ¿Los scores de la CNN se relacionan con la intensidad experimental?
                - ¿Qué regiones parecen más importantes para el modelo?
                - ¿El MSA/logo muestra un motivo claro?
                - Si compararon filtros con logo, ¿coinciden?

## Checklist final del TP5

                - [ ] Dataset positivo/negativo documentado.
                - [ ] OHE con forma `(N, 4, nt_len)` validado.
                - [ ] Split entrenamiento/validación documentado.
                - [ ] CNN base definida y probada sin entrenar.
                - [ ] Entrenamientos de 10, 50 y 100 épocas comparados.
                - [ ] Métricas: LF, matriz de confusión, Accuracy, F1, ROC/PR y AUC.
                - [ ] Exploración de filtros y tamaños de kernel.
                - [ ] Visualización de filtros entrenados.
                - [ ] Experimento con MaxPooling.
                - [ ] Análisis biológico con intensidad de pico, masking, MSA/logo.
                - [ ] Figuras/resultados guardados en `results/`.
                - [ ] Conclusiones escritas por sección.